In [6]:
import pandas as pd
from src.config import settings
from src.fs_io.dataframes import read_parquet
from langchain_openai import AzureChatOpenAI
from dotenv import load_dotenv

import os
load_dotenv()

True

In [15]:
llm_gen = AzureChatOpenAI(
    azure_deployment="gpt-5-mini",
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
)

chunks_df = read_parquet(settings.CHUNKS_DF_PATH)
sample_chunks = chunks_df.sample(50)

eval_data = []

print("Generating questions...")
for _, row in sample_chunks.iterrows():
    context = row['text'][:1000]
    prompt = f"Сформулюй одне питання українською мовою, відповідь на яке є в цьому тексті:\n{context}"

    try:
        q = llm_gen.invoke(prompt).content.strip()
        eval_data.append({
            "query": q,
            "expected_id": row['id'],
            "ground_truth_text": row['text']
        })
    except Exception as e:
        print(f"Error: {e}")

df_eval = pd.DataFrame(eval_data)
df_eval.head()

,query,expected_id,ground_truth_text
0,Скільки повстань у таборах ГУЛАГу відбулося у ...,1667,Українці в повстаннях у таборах ГУЛАГу. У пово...
1,У якому стилі було зведено Володимирський собо...,1182,Володимирський собор у Києві було зведено в не...
2,У якому році Кримське ханство стало васалом Ос...,1382,"Унаслідок розпаду Галицько-Волинської держави,..."
3,Які категорії селян стали найбіднішою частиною...,840,Поширення фільварків на українських землях при...
4,Що спричинило пожвавлення просвітницької діяль...,1117,"ІСТОРИЧНЕ ДЖЕРЕЛО 2. Установіть, про що йдетьс..."


In [17]:
df_eval.head(50)

,query,expected_id,ground_truth_text
0,Скільки повстань у таборах ГУЛАГу відбулося у ...,1667,Українці в повстаннях у таборах ГУЛАГу. У пово...
1,У якому стилі було зведено Володимирський собо...,1182,Володимирський собор у Києві було зведено в не...
2,У якому році Кримське ханство стало васалом Ос...,1382,"Унаслідок розпаду Галицько-Волинської держави,..."
3,Які категорії селян стали найбіднішою частиною...,840,Поширення фільварків на українських землях при...
4,Що спричинило пожвавлення просвітницької діяль...,1117,"ІСТОРИЧНЕ ДЖЕРЕЛО 2. Установіть, про що йдетьс..."
5,Хто є засновником кібернетики?,357,"y y Інформація — це новини, факти, знання, отр..."
6,"Як називали людину, яка керувала спорудженням ...",1221,Ніл тече від кордонів Ефіопії по прямій лінії ...
7,У чому полягає головна відмінність між спільно...,352,Головна відмінність між спільнотами в минулому...
8,Яку угоду з Європейським Союзом було підписано...,673,збереження державного суверенітету України лік...
9,Які раднаргоспи були розукрупнені в травні 196...,494,розташовувалися ради народного господарства. У...


In [16]:
df_eval.to_parquet(settings.DFS_PATH / 'eval_texts.parquet')

In [19]:
images_df = read_parquet(settings.IMAGES_DF_PATH)

valid_images = images_df[
    (images_df['caption'].str.len() > 20) &
    (images_df['caption'] != 'Зображення без опису')
].copy()

sample_images = valid_images.sample(min(50, len(valid_images)))

image_eval_data = []

print(f"Generating queries for {len(sample_images)} images...")

for _, row in sample_images.iterrows():
    caption = row['caption']

    prompt = f"""
    Опис зображення з підручника історії:
    "{caption}"

    Твоє завдання: Сформулюй питання учня, відповідь на яке стосується події, особи чи об'єкта з цього опису.

    Правила:
    1. НЕ вживай слова "фото", "зображення", "картинка", "покажи".
    2. Питання має бути про історію (хто це? що відбувається? коли це було?).
    3. Уяви, що це зображення буде ідеальним доповненням до відповіді на це питання.

    Приклад:
    Опис: "Богдан Хмельницький в'їжджає до Києва, 1648 рік"
    Питання: "Як зустрічали гетьмана після перших перемог Національно-визвольної війни?"

    Твоє питання:
    """

    try:
        q = llm_gen.invoke(prompt).content.strip()

        image_eval_data.append({
            "query": q,
            "expected_image_path": row['path'],
            "expected_doc_id": row['doc_id'],
            "ground_truth_caption": caption
        })
        print(f"+ {q}")
    except Exception as e:
        print(f"Error: {e}")

+ Яка історична подія відтворена на цій сцені і чому вона мала значення для подальшого розвитку регіону?


In [20]:
df_image_eval = pd.DataFrame(image_eval_data)
df_image_eval.head()

,query,expected_image_path,expected_doc_id,ground_truth_caption
0,Хто така Катерина Поліщук («Пташка») і яку рол...,/home/serpanok/Documents/agi/projects/UA_histo...,10,Катерина Поліщук — волонтерка і парамедикиня з...
1,Які риси характеру М. Хрущова й які події чи я...,/home/serpanok/Documents/agi/projects/UA_histo...,3,Проаналізуйте візуальні джерела. Зважаючи на о...
2,Як і коли в Україні проводили механізацію ручн...,/home/serpanok/Documents/agi/projects/UA_histo...,3,"– Як бачите, і ми механізували ручну працю."
3,Коли було збудовано Вірменський собор у Львові...,/home/serpanok/Documents/agi/projects/UA_histo...,1,Вірменський собор у Львові і його розписи. Суч...
4,Коли й між ким було укладено Зборівський мирни...,/home/serpanok/Documents/agi/projects/UA_histo...,9,1649 р. — Зборівський мирний договір


In [21]:
df_image_eval.to_parquet(settings.DFS_PATH / 'eval_images.parquet')